# Stage 1: House Price Enrichment Ingestion Layer

This section prepares the UK House Price Index enrichment dataset for ingestion into Snowflake.  
The original dataset was too large and wide for direct upload, so it was processed locally in Python before being loaded into the Snowflake Bronze layer.

## 1. Import Required Libraries

The pipeline uses pandas for CSV processing and glob to locate the split source files.

## 2. Locate Split House Price Files

The original house price CSV was split into seven smaller files to support scalable batch processing and avoid upload limits.

## 3. Read Files in Batches

Each split CSV file is read iteratively rather than loading the full source file in one step.  
This supports scalable ingestion and reduces memory pressure.

## 4. Remove Empty and Unnecessary Columns

Completely empty columns are removed.  

## 5. Clean House Price Values

House price values are cleaned by removing formatting issues, converting values to numeric format, and removing rows where no valid price is available.

## 6. Combine Cleaned Batches

All cleaned batches are combined into one dataframe. Duplicate rows are removed to improve data quality before aggregation.

## 7. Aggregate to Local Authority × Period Grain

The cleaned data is aggregated to local authority and period level.  
This creates a smaller, more suitable enrichment dataset for Snowflake ingestion and later police force mapping.

## 8. Export Bronze Ingestion File

The processed enrichment dataset is exported as a CSV file for upload into the Snowflake Bronze layer.

In [3]:
import pandas as pd
import glob
import os

files = sorted(glob.glob("split_house_prices/house_prices_part_*.csv"))

print("Files found:", len(files))

clean_frames = []

for file in files:
    print("Reading:", file)

    df = pd.read_csv(file, dtype=str, low_memory=False)

    # Drop completely empty columns only
    df = df.dropna(axis=1, how="all")

    # KEEP LSOA columns this time
    id_cols = [
        "Local authority code",
        "Local authority name",
        "LSOA code",
        "LSOA name"
    ]

    value_cols = [c for c in df.columns if c not in id_cols]

    df_long = df.melt(
        id_vars=id_cols,
        value_vars=value_cols,
        var_name="period",
        value_name="average_house_price"
    )

    df_long["average_house_price"] = (
        df_long["average_house_price"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    df_long["average_house_price"] = pd.to_numeric(
        df_long["average_house_price"],
        errors="coerce"
    )

    df_long = df_long.dropna(subset=["average_house_price"])

    clean_frames.append(df_long)

house_lsoa = pd.concat(clean_frames, ignore_index=True)

house_lsoa = (
    house_lsoa
    .groupby(
        [
            "Local authority code",
            "Local authority name",
            "LSOA code",
            "LSOA name",
            "period"
        ],
        as_index=False
    )
    .agg(
        average_house_price=("average_house_price", "mean"),
        records_used=("average_house_price", "count")
    )
)

house_lsoa.to_csv("house_prices_lsoa_period.csv", index=False)

print("Saved: house_prices_lsoa_period.csv")
print("Rows:", len(house_lsoa))
print(house_lsoa.head())

Files found: 7
Reading: split_house_prices/house_prices_part_1.csv
Reading: split_house_prices/house_prices_part_2.csv
Reading: split_house_prices/house_prices_part_3.csv
Reading: split_house_prices/house_prices_part_4.csv
Reading: split_house_prices/house_prices_part_5.csv
Reading: split_house_prices/house_prices_part_6.csv
Reading: split_house_prices/house_prices_part_7.csv
Saved: house_prices_lsoa_period.csv
Rows: 3741179
  Local authority code Local authority name  LSOA code        LSOA name  \
0            E06000001           Hartlepool  E01011949  Hartlepool 009A   
1            E06000001           Hartlepool  E01011949  Hartlepool 009A   
2            E06000001           Hartlepool  E01011949  Hartlepool 009A   
3            E06000001           Hartlepool  E01011949  Hartlepool 009A   
4            E06000001           Hartlepool  E01011949  Hartlepool 009A   

                 period  average_house_price  records_used  
0  Year ending Dec 1995              34750.0             1 

- Local Jupyter pre-processing: raw 7 CSV chunks → reduced upload file

In [4]:
selected_authorities = [
    # West Midlands
    "Birmingham", "Coventry", "Dudley", "Sandwell", "Solihull", "Walsall", "Wolverhampton",

    # Surrey
    "Elmbridge", "Epsom and Ewell", "Guildford", "Mole Valley", "Reigate and Banstead",
    "Runnymede", "Spelthorne", "Surrey Heath", "Tandridge", "Waverley", "Woking",

    # Thames Valley
    "Bracknell Forest", "Reading", "Slough", "West Berkshire", "Windsor and Maidenhead",
    "Wokingham", "Aylesbury Vale", "Buckingham", "Chiltern", "Milton Keynes",
    "South Bucks", "Wycombe", "Cherwell", "Oxford", "South Oxfordshire",
    "Vale of White Horse", "West Oxfordshire",

    # Dyfed-Powys
    "Carmarthenshire", "Ceredigion", "Pembrokeshire", "Powys"
]

house_lsoa_filtered = house_lsoa[
    house_lsoa["Local authority name"].isin(selected_authorities)
].copy()

house_lsoa_filtered.to_csv("house_prices_lsoa_selected_forces.csv", index=False)

print("Saved: house_prices_lsoa_selected_forces.csv")
print("Rows:", len(house_lsoa_filtered))
print(house_lsoa_filtered["Local authority name"].nunique())
print(house_lsoa_filtered["Local authority name"].sort_values().unique())

Saved: house_prices_lsoa_selected_forces.csv
Rows: 407981
34
['Birmingham' 'Bracknell Forest' 'Carmarthenshire' 'Ceredigion' 'Cherwell'
 'Coventry' 'Dudley' 'Elmbridge' 'Epsom and Ewell' 'Guildford'
 'Milton Keynes' 'Mole Valley' 'Oxford' 'Pembrokeshire' 'Powys' 'Reading'
 'Reigate and Banstead' 'Runnymede' 'Sandwell' 'Slough' 'Solihull'
 'South Oxfordshire' 'Spelthorne' 'Surrey Heath' 'Tandridge'
 'Vale of White Horse' 'Walsall' 'Waverley' 'West Berkshire'
 'West Oxfordshire' 'Windsor and Maidenhead' 'Woking' 'Wokingham'
 'Wolverhampton']
